In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import h5py
from utils import preprocess, calFlow3d_Wei_v1, visualization,mask,option,IO,reference
from utils import registration
import numpy as np
import nd2
import tifffile as tiff
from matplotlib import pyplot as plt
import cupy as cp

#filePath
movingFilePath="/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15849.nd2"
refenceFilePath_start='/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15846.nd2'
refenceFilePath_end='/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15850.nd2'
config_file='/home/cyf/wbi/Virginia/wbi_0811/wholistic_registration/config.toml'
base_folder="/home/cyf/wbi/Virginia/f2013_0814registraed/"
os.makedirs(base_folder, exist_ok=True)

mem_folder = os.path.join(base_folder, "membrane")
ca_folder = os.path.join(base_folder, "ca")
concat_folder= os.path.join(base_folder, "concat")

os.makedirs(mem_folder, exist_ok=True)
os.makedirs(ca_folder, exist_ok=True)
os.makedirs(concat_folder, exist_ok=True)
#read meta data
meta_start=IO.readMeta(refenceFilePath_start)
meta_end=IO.readMeta(refenceFilePath_end)
meta_moving=IO.readMeta(movingFilePath)

#process 2000 frames in 1 iteration
total_frames = 47997
chunk_size = 10000

CuPy is available with CUDA - using GPU acceleration
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 19)
Total frames is 2
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 19)
Total frames is 3
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 1)
Total frames is 47997


In [2]:
mem_data=IO.readFrame(movingFilePath, range(20000, 30000,10), channel=1)
Ca_data=IO.readFrame(movingFilePath, range(20000, 30000,10), channel=0)
frames = range(20000, 30000,10)
#pick reference
mem_ref,indsort=reference.pick_initial_reference(mem_data)
Ca_data_reshape=np.reshape(Ca_data, (len(frames), -1))
Ca_average=np.mean(Ca_data_reshape[indsort,:],axis=0)
Ca_ref_1plane=np.reshape(Ca_average,(mem_data.shape[1],mem_data.shape[2]))
Ca_ref_1plane_transform=300*np.log10(1+Ca_ref_1plane)
ref_img=mem_ref+Ca_ref_1plane_transform

for start_frame in range(0, total_frames, chunk_size):
    end_frame = min(start_frame + chunk_size, total_frames)
    print(f"Processing frames {start_frame} ~ {end_frame-1}")

    # read moving images
    frames = range(start_frame, end_frame,10)
    mem_data = IO.readFrame(movingFilePath, frames, channel=1)
    Ca_data  = IO.readFrame(movingFilePath, frames, channel=0)


    # do regitration
    mem_channel, Ca_channel, concat_images, errors = registration.wbi_registration_2d(
        mem_data,
        Ca_data,
        config_file,
        ref_img
    )

    # generate the name of the files
    memPath     = os.path.join(mem_folder, f"membrane_corrected_{start_frame}~{end_frame}_0822.tiff")
    CaPath      = os.path.join(ca_folder, f"Ca_corrected_{start_frame}~{end_frame}_0822.tiff")
    concatPath  = os.path.join(concat_folder, f"Concat_{start_frame}~{end_frame}_0822.tiff")

    # save results
    IO.saveTiff(mem_channel,   config_file, memPath)
    IO.saveTiff(Ca_channel,    config_file, CaPath)
    IO.saveTiff(concat_images, config_file, concatPath)

    print(f"Saved: {memPath}, {CaPath}, {concatPath}")

print("All frames processed.")

Processing frames 0 ~ 9999
Frame: 1	Initial Error is:945.1160278320312	Eventual Error: 489.1518249511719
Frame: 2	Initial Error is:905.1366577148438	Eventual Error: 484.8215026855469
Frame: 3	Initial Error is:876.6807250976562	Eventual Error: 477.3301086425781
Frame: 4	Initial Error is:874.8260498046875	Eventual Error: 491.4436340332031
Frame: 5	Initial Error is:860.9354858398438	Eventual Error: 481.984375
Frame: 6	Initial Error is:886.3616943359375	Eventual Error: 488.0934143066406
Frame: 7	Initial Error is:998.1958618164062	Eventual Error: 491.2521667480469
Frame: 8	Initial Error is:887.9794921875	Eventual Error: 481.379150390625
Frame: 9	Initial Error is:943.6846923828125	Eventual Error: 475.0423889160156
Frame: 10	Initial Error is:904.0286865234375	Eventual Error: 487.1278076171875
Frame: 11	Initial Error is:914.46923828125	Eventual Error: 486.785400390625
Frame: 12	Initial Error is:965.036865234375	Eventual Error: 486.41241455078125
Frame: 13	Initial Error is:940.0079956054688	Eve

/home/cyf/.conda/envs/wbi/lib/python3.10/site-packages/tifffile/tifffile.py:2310: UserWarning: <tifffile.TiffWriter 'Concat_0~10000_0822.tiff'> writing zero-size array to nonconformant TIFF
  warnings.warn(


Saved: /home/cyf/wbi/Virginia/f2013_0814registraed/membrane/membrane_corrected_0~10000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/ca/Ca_corrected_0~10000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/concat/Concat_0~10000_0822.tiff
Processing frames 10000 ~ 19999
Frame: 1	Initial Error is:914.8298950195312	Eventual Error: 437.70806884765625
Frame: 2	Initial Error is:892.0205078125	Eventual Error: 430.5389709472656
Frame: 3	Initial Error is:833.5340576171875	Eventual Error: 423.90167236328125
Frame: 4	Initial Error is:864.6196899414062	Eventual Error: 421.3280944824219
Frame: 5	Initial Error is:854.2600708007812	Eventual Error: 429.364990234375
Frame: 6	Initial Error is:824.3082275390625	Eventual Error: 428.3594665527344
Frame: 7	Initial Error is:873.7454223632812	Eventual Error: 422.63128662109375
Frame: 8	Initial Error is:816.6630859375	Eventual Error: 422.2664794921875
Frame: 9	Initial Error is:849.4114990234375	Eventual Error: 424.80584716796875
Frame: 10	Initia

/home/cyf/.conda/envs/wbi/lib/python3.10/site-packages/tifffile/tifffile.py:2310: UserWarning: <tifffile.TiffWriter 'Concat_10000~20000_0822.tiff'> writing zero-size array to nonconformant TIFF
  warnings.warn(


Saved: /home/cyf/wbi/Virginia/f2013_0814registraed/membrane/membrane_corrected_10000~20000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/ca/Ca_corrected_10000~20000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/concat/Concat_10000~20000_0822.tiff
Processing frames 20000 ~ 29999
Frame: 1	Initial Error is:1013.0242309570312	Eventual Error: 461.3840026855469
Frame: 2	Initial Error is:976.5948486328125	Eventual Error: 454.0408935546875
Frame: 3	Initial Error is:1017.5324096679688	Eventual Error: 451.3467712402344
Frame: 4	Initial Error is:943.3875122070312	Eventual Error: 452.64508056640625
Frame: 5	Initial Error is:997.5166625976562	Eventual Error: 447.9762878417969
Frame: 6	Initial Error is:1009.9879760742188	Eventual Error: 451.1971130371094
Frame: 7	Initial Error is:967.2210693359375	Eventual Error: 450.4686584472656
Frame: 8	Initial Error is:980.40478515625	Eventual Error: 451.04888916015625
Frame: 9	Initial Error is:944.7976684570312	Eventual Error: 438.490142822265

/home/cyf/.conda/envs/wbi/lib/python3.10/site-packages/tifffile/tifffile.py:2310: UserWarning: <tifffile.TiffWriter 'Concat_20000~30000_0822.tiff'> writing zero-size array to nonconformant TIFF
  warnings.warn(


Saved: /home/cyf/wbi/Virginia/f2013_0814registraed/membrane/membrane_corrected_20000~30000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/ca/Ca_corrected_20000~30000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/concat/Concat_20000~30000_0822.tiff
Processing frames 30000 ~ 39999
Frame: 1	Initial Error is:866.6909790039062	Eventual Error: 421.95318603515625
Frame: 2	Initial Error is:879.0507202148438	Eventual Error: 422.20556640625
Frame: 3	Initial Error is:903.2593383789062	Eventual Error: 440.0845642089844
Absolute difference between old and new error is less than 1e-3
Frame: 4	Initial Error is:877.4939575195312	Eventual Error: 424.3018798828125
Frame: 5	Initial Error is:893.8815307617188	Eventual Error: 425.8652648925781
Frame: 6	Initial Error is:878.8224487304688	Eventual Error: 413.0458679199219
Frame: 7	Initial Error is:869.8800048828125	Eventual Error: 419.7328186035156
Frame: 8	Initial Error is:884.4898681640625	Eventual Error: 428.2080383300781
Frame: 9	Initial

/home/cyf/.conda/envs/wbi/lib/python3.10/site-packages/tifffile/tifffile.py:2310: UserWarning: <tifffile.TiffWriter 'Concat_30000~40000_0822.tiff'> writing zero-size array to nonconformant TIFF
  warnings.warn(


Saved: /home/cyf/wbi/Virginia/f2013_0814registraed/membrane/membrane_corrected_30000~40000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/ca/Ca_corrected_30000~40000_0822.tiff, /home/cyf/wbi/Virginia/f2013_0814registraed/concat/Concat_30000~40000_0822.tiff
Processing frames 40000 ~ 47996
Frame: 1	Initial Error is:904.6887817382812	Eventual Error: 470.70648193359375
Frame: 2	Initial Error is:928.6957397460938	Eventual Error: 481.0625
Frame: 3	Initial Error is:875.7677612304688	Eventual Error: 466.1436462402344
Frame: 4	Initial Error is:918.3865966796875	Eventual Error: 476.9807434082031
Frame: 5	Initial Error is:931.7150268554688	Eventual Error: 473.78607177734375
Frame: 6	Initial Error is:923.982666015625	Eventual Error: 471.85430908203125
Frame: 7	Initial Error is:904.509765625	Eventual Error: 479.485107421875
Frame: 8	Initial Error is:896.5311889648438	Eventual Error: 474.3085021972656
Frame: 9	Initial Error is:942.5948486328125	Eventual Error: 494.50128173828125
Frame: 10	In

/home/cyf/.conda/envs/wbi/lib/python3.10/site-packages/tifffile/tifffile.py:2310: UserWarning: <tifffile.TiffWriter 'Concat_40000~47997_0822.tiff'> writing zero-size array to nonconformant TIFF
  warnings.warn(


View  zarr
---

In [ ]:
import zarr
import matplotlib.pyplot as plt
import ipywidgets as widgets
from utils import visualization
z = zarr.open("/home/cyf/wbi/Virginia/f2013_0814registraed/concat/Concat_6998~8998_aligned.zarr", mode="r")

mem = z["membrane"]
cal = z["calcium"]
ref = z["reference"]

def show_frame(i, dataset="membrane"):
    data = z[dataset]
    plt.figure(figsize=(10,10))
    plt.imshow(visualization.auto_contrast(data[i]), cmap="gray")
    plt.title(f"{dataset} - frame {i}")
    plt.axis("off")
    plt.show()

widgets.interact(show_frame, 
                 i=(0, mem.shape[0]-1), 
                 dataset=["membrane", "calcium", "reference"])
